In [14]:
import pandas as pd
import numpy as np

games = pd.read_csv("../data/processed/games_cleaned.csv")

games["GAME_DATE_EST"] = pd.to_datetime(games["GAME_DATE_EST"])

games.head()

,GAME_DATE_EST,GAME_ID,GAME_STATUS_TEXT,HOME_TEAM_ID,VISITOR_TEAM_ID,SEASON,TEAM_ID_home,PTS_home,FG_PCT_home,FT_PCT_home,...,REB_home,TEAM_ID_away,PTS_away,FG_PCT_away,FT_PCT_away,FG3_PCT_away,AST_away,REB_away,HOME_TEAM_WINS,TOTAL_POINTS
0,2003-10-05,10300001,Final,1610612762,1610612742,2003,1610612762,90.0,0.457,0.735,...,41.0,1610612742,85.0,0.447,0.500,0.250,20.0,38.0,1,175.0
1,2003-10-06,10300002,Final,1610612763,1610612749,2003,1610612763,105.0,0.494,0.618,...,48.0,1610612749,94.0,0.427,0.700,0.154,20.0,43.0,1,199.0
2,2003-10-07,10300009,Final,1610612758,1610612746,2003,1610612758,101.0,0.467,0.871,...,39.0,1610612746,82.0,0.368,0.609,0.364,13.0,50.0,1,183.0
3,2003-10-07,10300005,Final,1610612757,1610612745,2003,1610612757,104.0,0.527,0.657,...,33.0,1610612745,80.0,0.470,0.667,0.333,10.0,37.0,1,184.0
4,2003-10-07,10300007,Final,1610612748,1610612755,2003,1610612748,86.0,0.352,0.647,...,55.0,1610612755,79.0,0.329,0.897,0.143,7.0,44.0,1,165.0


In [15]:
games["GAME_ID"].duplicated().sum()

np.int64(0)

In [16]:
#previos 3 games team statistics

In [17]:
#create HOME team records

In [18]:
home = games[[
    "GAME_ID",
    "GAME_DATE_EST",
    "SEASON",
    "TEAM_ID_home",
    "TEAM_ID_away",
    "PTS_home",
    "PTS_away",
    "FG_PCT_home",
    "FT_PCT_home",
    "FG3_PCT_home",
    "AST_home",
    "REB_home"
]].copy()

In [19]:
home.columns = [
    "GAME_ID",
    "GAME_DATE_EST",
    "SEASON",
    "TEAM_ID",
    "OPPONENT_ID",
    "POINTS_SCORED",
    "POINTS_ALLOWED",
    "FG_PCT",
    "FT_PCT",
    "FG3_PCT",
    "AST",
    "REB"
]

In [20]:
home["IS_HOME"] = 1

In [21]:
#create AWAY team records

In [22]:
away = games[[
    "GAME_ID",
    "GAME_DATE_EST",
    "SEASON",
    "TEAM_ID_away",
    "TEAM_ID_home",
    "PTS_away",
    "PTS_home",
    "FG_PCT_away",
    "FT_PCT_away",
    "FG3_PCT_away",
    "AST_away",
    "REB_away"
]].copy()

In [23]:
away.columns = [
    "GAME_ID",
    "GAME_DATE_EST",
    "SEASON",
    "TEAM_ID",
    "OPPONENT_ID",
    "POINTS_SCORED",
    "POINTS_ALLOWED",
    "FG_PCT",
    "FT_PCT",
    "FG3_PCT",
    "AST",
    "REB"
]

In [24]:
away["IS_HOME"] = 0

In [25]:
team_games = pd.concat([home, away], ignore_index=True)

In [26]:

team_games = team_games.sort_values(
    ["TEAM_ID", "GAME_DATE_EST"]
).reset_index(drop=True)

In [27]:
team_games.head(10)

,GAME_ID,GAME_DATE_EST,SEASON,TEAM_ID,OPPONENT_ID,POINTS_SCORED,POINTS_ALLOWED,FG_PCT,FT_PCT,FG3_PCT,AST,REB,IS_HOME
0,10300011,2003-10-08,2003,1610612737,1610612739,80.0,83.0,0.350,0.750,0.429,18.0,45.0,1
1,20300006,2003-10-29,2003,1610612737,1610612740,83.0,88.0,0.398,0.737,0.214,18.0,58.0,0
2,20300024,2003-10-31,2003,1610612737,1610612741,94.0,100.0,0.400,0.714,0.583,22.0,48.0,0
3,20300029,2003-11-01,2003,1610612737,1610612754,99.0,103.0,0.468,0.677,0.375,23.0,36.0,1
4,20300042,2003-11-03,2003,1610612737,1610612740,90.0,80.0,0.427,0.652,0.333,20.0,50.0,1
5,20300060,2003-11-05,2003,1610612737,1610612744,72.0,99.0,0.367,0.500,0.333,19.0,43.0,0
6,20300084,2003-11-08,2003,1610612737,1610612757,83.0,90.0,0.438,0.786,0.100,21.0,45.0,0
7,20300089,2003-11-09,2003,1610612737,1610612760,91.0,81.0,0.479,0.789,0.533,16.0,41.0,0
8,20300102,2003-11-11,2003,1610612737,1610612746,103.0,115.0,0.543,0.952,0.389,25.0,30.0,0
9,20300109,2003-11-12,2003,1610612737,1610612756,99.0,94.0,0.457,0.833,0.476,19.0,44.0,0


In [28]:
team_games.shape

(59254, 13)

In [29]:
team_games["TEAM_ID"].nunique()

30

In [30]:
feature_cols = [
    "POINTS_SCORED",
    "POINTS_ALLOWED",
    "FG_PCT",
    "FT_PCT",
    "FG3_PCT",
    "AST",
    "REB"
]

for col in feature_cols:
    team_games[f"LAST3_{col}"] = (
        team_games
        .groupby("TEAM_ID")[col]
        .transform(lambda x: x.shift(1).rolling(3).mean())
    )

In [31]:
team_games[[
    "TEAM_ID",
    "GAME_DATE_EST",
    "POINTS_SCORED",
    "LAST3_POINTS_SCORED",
    "POINTS_ALLOWED",
    "LAST3_POINTS_ALLOWED"
]].head(10)

,TEAM_ID,GAME_DATE_EST,POINTS_SCORED,LAST3_POINTS_SCORED,POINTS_ALLOWED,LAST3_POINTS_ALLOWED
0,1610612737,2003-10-08,80.0,NaN,83.0,NaN
1,1610612737,2003-10-29,83.0,NaN,88.0,NaN
2,1610612737,2003-10-31,94.0,NaN,100.0,NaN
3,1610612737,2003-11-01,99.0,85.666667,103.0,90.333333
4,1610612737,2003-11-03,90.0,92.000000,80.0,97.000000
5,1610612737,2003-11-05,72.0,94.333333,99.0,94.333333
6,1610612737,2003-11-08,83.0,87.000000,90.0,94.000000
7,1610612737,2003-11-09,91.0,81.666667,81.0,89.666667
8,1610612737,2003-11-11,103.0,82.000000,115.0,90.000000
9,1610612737,2003-11-12,99.0,92.333333,94.0,95.333333


In [32]:
team_id = team_games["TEAM_ID"].iloc[0]

In [33]:
team_games[
    team_games["TEAM_ID"] == team_id
][[
    "GAME_DATE_EST",
    "POINTS_SCORED",
    "LAST3_POINTS_SCORED"
]].head(8)

,GAME_DATE_EST,POINTS_SCORED,LAST3_POINTS_SCORED
0,2003-10-08,80.0,NaN
1,2003-10-29,83.0,NaN
2,2003-10-31,94.0,NaN
3,2003-11-01,99.0,85.666667
4,2003-11-03,90.0,92.000000
5,2003-11-05,72.0,94.333333
6,2003-11-08,83.0,87.000000
7,2003-11-09,91.0,81.666667


In [34]:
#perfromance difference between the team and the opponent in the last 3 games (PLUS/MINUS)

In [35]:
team_games["POINT_DIFFERENTIAL"] = (
    team_games["POINTS_SCORED"] -
    team_games["POINTS_ALLOWED"]
)

In [36]:
team_games["LAST3_POINT_DIFFERENTIAL"] = (
    team_games
    .groupby("TEAM_ID")["POINT_DIFFERENTIAL"]
    .transform(lambda x: x.shift(1).rolling(3).mean())
)

In [37]:
#checking the last game played/ no of rest days

In [38]:
team_games["DAYS_SINCE_LAST_GAME"] = (
    team_games
    .groupby("TEAM_ID")["GAME_DATE_EST"]
    .diff()
    .dt.days
)

In [39]:
team_games[[
    "TEAM_ID",
    "GAME_DATE_EST",
    "POINTS_SCORED",
    "LAST3_POINTS_SCORED",
    "LAST3_POINTS_ALLOWED",
    "LAST3_FG_PCT",
    "LAST3_FT_PCT",
    "LAST3_FG3_PCT",
    "LAST3_AST",
    "LAST3_REB",
    "LAST3_POINT_DIFFERENTIAL",
    "DAYS_SINCE_LAST_GAME"
]].head(10)

,TEAM_ID,GAME_DATE_EST,POINTS_SCORED,LAST3_POINTS_SCORED,LAST3_POINTS_ALLOWED,LAST3_FG_PCT,LAST3_FT_PCT,LAST3_FG3_PCT,LAST3_AST,LAST3_REB,LAST3_POINT_DIFFERENTIAL,DAYS_SINCE_LAST_GAME
0,1610612737,2003-10-08,80.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1610612737,2003-10-29,83.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.0
2,1610612737,2003-10-31,94.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
3,1610612737,2003-11-01,99.0,85.666667,90.333333,0.382667,0.733667,0.408667,19.333333,50.333333,-4.666667,1.0
4,1610612737,2003-11-03,90.0,92.000000,97.000000,0.422000,0.709333,0.390667,21.000000,47.333333,-5.000000,2.0
5,1610612737,2003-11-05,72.0,94.333333,94.333333,0.431667,0.681000,0.430333,21.666667,44.666667,0.000000,2.0
6,1610612737,2003-11-08,83.0,87.000000,94.000000,0.420667,0.609667,0.347000,20.666667,43.000000,-7.000000,3.0
7,1610612737,2003-11-09,91.0,81.666667,89.666667,0.410667,0.646000,0.255333,20.000000,46.000000,-8.000000,1.0
8,1610612737,2003-11-11,103.0,82.000000,90.000000,0.428000,0.691667,0.322000,18.666667,43.000000,-8.000000,2.0
9,1610612737,2003-11-12,99.0,92.333333,95.333333,0.486667,0.842333,0.340667,20.666667,38.666667,-3.000000,1.0


In [40]:
feature_cols = [
    "POINTS_SCORED",
    "POINTS_ALLOWED",
    "FG_PCT",
    "FT_PCT",
    "FG3_PCT",
    "AST",
    "REB"
]

for col in feature_cols:
    team_games[f"LAST3_{col}"] = (
        team_games
        .groupby("TEAM_ID")[col]
        .transform(lambda x: x.shift(1).rolling(3).mean())
    )

In [41]:
team_games["POINT_DIFFERENTIAL"] = (
    team_games["POINTS_SCORED"] -
    team_games["POINTS_ALLOWED"]
)

team_games["LAST3_POINT_DIFFERENTIAL"] = (
    team_games
    .groupby("TEAM_ID")["POINT_DIFFERENTIAL"]
    .transform(lambda x: x.shift(1).rolling(3).mean())
)

In [42]:
team_games["DAYS_SINCE_LAST_GAME"] = (
    team_games
    .groupby("TEAM_ID")["GAME_DATE_EST"]
    .diff()
    .dt.days
)

In [43]:
team_games[[
    "TEAM_ID",
    "GAME_DATE_EST",
    "POINTS_SCORED",
    "LAST3_POINTS_SCORED",
    "LAST3_POINTS_ALLOWED",
    "LAST3_FG_PCT",
    "LAST3_FT_PCT",
    "LAST3_FG3_PCT",
    "LAST3_AST",
    "LAST3_REB",
    "LAST3_POINT_DIFFERENTIAL",
    "DAYS_SINCE_LAST_GAME"
]].head(10)

,TEAM_ID,GAME_DATE_EST,POINTS_SCORED,LAST3_POINTS_SCORED,LAST3_POINTS_ALLOWED,LAST3_FG_PCT,LAST3_FT_PCT,LAST3_FG3_PCT,LAST3_AST,LAST3_REB,LAST3_POINT_DIFFERENTIAL,DAYS_SINCE_LAST_GAME
0,1610612737,2003-10-08,80.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1610612737,2003-10-29,83.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.0
2,1610612737,2003-10-31,94.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
3,1610612737,2003-11-01,99.0,85.666667,90.333333,0.382667,0.733667,0.408667,19.333333,50.333333,-4.666667,1.0
4,1610612737,2003-11-03,90.0,92.000000,97.000000,0.422000,0.709333,0.390667,21.000000,47.333333,-5.000000,2.0
5,1610612737,2003-11-05,72.0,94.333333,94.333333,0.431667,0.681000,0.430333,21.666667,44.666667,0.000000,2.0
6,1610612737,2003-11-08,83.0,87.000000,94.000000,0.420667,0.609667,0.347000,20.666667,43.000000,-7.000000,3.0
7,1610612737,2003-11-09,91.0,81.666667,89.666667,0.410667,0.646000,0.255333,20.000000,46.000000,-8.000000,1.0
8,1610612737,2003-11-11,103.0,82.000000,90.000000,0.428000,0.691667,0.322000,18.666667,43.000000,-8.000000,2.0
9,1610612737,2003-11-12,99.0,92.333333,95.333333,0.486667,0.842333,0.340667,20.666667,38.666667,-3.000000,1.0


In [44]:
history_features = [
    "LAST3_POINTS_SCORED",
    "LAST3_POINTS_ALLOWED",
    "LAST3_FG_PCT",
    "LAST3_FT_PCT",
    "LAST3_FG3_PCT",
    "LAST3_AST",
    "LAST3_REB",
    "LAST3_POINT_DIFFERENTIAL",
    "DAYS_SINCE_LAST_GAME"
]

In [45]:
home_features = team_games[team_games["IS_HOME"] == 1][
    ["GAME_ID", "TEAM_ID"] + history_features
].copy()

In [46]:
home_features = home_features.rename(columns={
    "TEAM_ID": "HOME_TEAM_ID"
})

In [47]:
home_features = home_features.rename(
    columns={
        col: f"HOME_{col}" for col in history_features
    }
)

In [48]:
away_features = team_games[team_games["IS_HOME"] == 0][
    ["GAME_ID", "TEAM_ID"] + history_features
].copy()

In [49]:
away_features = away_features.rename(columns={
    "TEAM_ID": "AWAY_TEAM_ID"
})

In [50]:
away_features = away_features.rename(
    columns={
        col: f"AWAY_{col}" for col in history_features
    }
)

In [51]:
model_data = games.merge(
    home_features,
    on=["GAME_ID", "HOME_TEAM_ID"],
    how="left"
)

In [56]:
model_data = model_data.merge(
    away_features,
    on="GAME_ID",
    how="left"
)


In [ ]:
model_data.shape

(29627, 31)

In [57]:
model_data.columns.tolist()

['GAME_DATE_EST',
 'GAME_ID',
 'GAME_STATUS_TEXT',
 'HOME_TEAM_ID',
 'VISITOR_TEAM_ID',
 'SEASON',
 'TEAM_ID_home',
 'PTS_home',
 'FG_PCT_home',
 'FT_PCT_home',
 'FG3_PCT_home',
 'AST_home',
 'REB_home',
 'TEAM_ID_away',
 'PTS_away',
 'FG_PCT_away',
 'FT_PCT_away',
 'FG3_PCT_away',
 'AST_away',
 'REB_away',
 'HOME_TEAM_WINS',
 'TOTAL_POINTS',
 'HOME_LAST3_POINTS_SCORED',
 'HOME_LAST3_POINTS_ALLOWED',
 'HOME_LAST3_FG_PCT',
 'HOME_LAST3_FT_PCT',
 'HOME_LAST3_FG3_PCT',
 'HOME_LAST3_AST',
 'HOME_LAST3_REB',
 'HOME_LAST3_POINT_DIFFERENTIAL',
 'HOME_DAYS_SINCE_LAST_GAME',
 'AWAY_TEAM_ID',
 'AWAY_LAST3_POINTS_SCORED',
 'AWAY_LAST3_POINTS_ALLOWED',
 'AWAY_LAST3_FG_PCT',
 'AWAY_LAST3_FT_PCT',
 'AWAY_LAST3_FG3_PCT',
 'AWAY_LAST3_AST',
 'AWAY_LAST3_REB',
 'AWAY_LAST3_POINT_DIFFERENTIAL',
 'AWAY_DAYS_SINCE_LAST_GAME']

In [58]:
model_data[[
    "GAME_DATE_EST",
    "HOME_TEAM_ID",
    "AWAY_TEAM_ID",
    "HOME_LAST3_POINTS_SCORED",
    "AWAY_LAST3_POINTS_SCORED",
    "HOME_LAST3_POINTS_ALLOWED",
    "AWAY_LAST3_POINTS_ALLOWED",
    "TOTAL_POINTS"
]].head(10)

,GAME_DATE_EST,HOME_TEAM_ID,AWAY_TEAM_ID,HOME_LAST3_POINTS_SCORED,AWAY_LAST3_POINTS_SCORED,HOME_LAST3_POINTS_ALLOWED,AWAY_LAST3_POINTS_ALLOWED,TOTAL_POINTS
0,2003-10-05,1610612762,1610612742,NaN,NaN,NaN,NaN,175.0
1,2003-10-06,1610612763,1610612749,NaN,NaN,NaN,NaN,199.0
2,2003-10-07,1610612758,1610612746,NaN,NaN,NaN,NaN,183.0
3,2003-10-07,1610612757,1610612745,NaN,NaN,NaN,NaN,184.0
4,2003-10-07,1610612748,1610612755,NaN,NaN,NaN,NaN,165.0
5,2003-10-07,1610612756,1610612751,NaN,NaN,NaN,NaN,180.0
6,2003-10-07,1610612742,1610612753,NaN,NaN,NaN,NaN,188.0
7,2003-10-07,1610612765,1610612739,NaN,NaN,NaN,NaN,196.0
8,2003-10-07,1610612764,1610612752,NaN,NaN,NaN,NaN,190.0
9,2003-10-08,1610612754,1610612741,NaN,NaN,NaN,NaN,120.0


In [59]:
model_data = model_data.dropna(
    subset=[
        f"HOME_{col}" for col in history_features
    ] + [
        f"AWAY_{col}" for col in history_features
    ]
)

In [60]:
model_data.shape

(29575, 41)

In [61]:
model_data[[
    "GAME_ID",
    "GAME_DATE_EST",
    "HOME_TEAM_ID",
    "AWAY_TEAM_ID",
    "HOME_LAST3_POINTS_SCORED",
    "AWAY_LAST3_POINTS_SCORED",
    "HOME_LAST3_REB",
    "AWAY_LAST3_REB",
    "HOME_LAST3_POINT_DIFFERENTIAL",
    "AWAY_LAST3_POINT_DIFFERENTIAL",
    "TOTAL_POINTS"
]].head(10)

,GAME_ID,GAME_DATE_EST,HOME_TEAM_ID,AWAY_TEAM_ID,HOME_LAST3_POINTS_SCORED,AWAY_LAST3_POINTS_SCORED,HOME_LAST3_REB,AWAY_LAST3_REB,HOME_LAST3_POINT_DIFFERENTIAL,AWAY_LAST3_POINT_DIFFERENTIAL,TOTAL_POINTS
39,20300020,2003-10-31,1610612748,1610612765,78.333333,95.666667,46.000000,39.000000,-10.333333,3.000000,174.0
42,20300028,2003-11-01,1610612753,1610612765,93.666667,94.666667,39.000000,40.000000,-2.333333,8.333333,181.0
43,20300030,2003-11-01,1610612740,1610612738,97.666667,93.333333,45.666667,41.666667,4.666667,3.333333,187.0
46,20300032,2003-11-01,1610612749,1610612741,92.000000,77.333333,44.666667,43.666667,-1.000000,-7.666667,166.0
49,20300029,2003-11-01,1610612737,1610612754,85.666667,76.666667,50.333333,31.333333,-4.666667,-2.666667,202.0
50,20300027,2003-11-01,1610612764,1610612751,83.000000,90.666667,42.000000,43.333333,5.666667,14.000000,183.0
55,20300043,2003-11-03,1610612763,1610612759,91.666667,80.333333,44.000000,45.666667,5.666667,1.666667,168.0
56,20300046,2003-11-03,1610612762,1610612750,97.000000,76.333333,36.000000,39.333333,-4.333333,0.000000,181.0
57,20300041,2003-11-03,1610612753,1610612752,89.333333,81.000000,37.666667,40.666667,-3.666667,-10.666667,143.0
58,20300042,2003-11-03,1610612737,1610612740,92.000000,95.000000,47.333333,43.333333,-5.000000,4.666667,170.0


In [62]:
model_data.to_csv(
    "../data/processed/model_features.csv",
    index=False
)

In [63]:
model_data["SEASON"].value_counts().sort_index()


SEASON
2003    1237
2004    1359
2005    1432
2006    1419
2007    1411
2008    1425
2009    1424
2010    1422
2011    1104
2012    1420
2013    1427
2014    1418
2015    1416
2016    1405
2017    1382
2018    1378
2019    1241
2020    1220
2021    1389
2022    1384
2023    1383
2024     879
Name: count, dtype: int64

In [64]:
model_data[
    ["SEASON", "GAME_DATE_EST"]
].sort_values("GAME_DATE_EST").head(20)

,SEASON,GAME_DATE_EST
39,2003,2003-10-31
42,2003,2003-11-01
43,2003,2003-11-01
46,2003,2003-11-01
49,2003,2003-11-01
50,2003,2003-11-01
55,2003,2003-11-03
56,2003,2003-11-03
57,2003,2003-11-03
58,2003,2003-11-03


In [65]:
#train/validation/test split

In [66]:
train_data = model_data[
    model_data["SEASON"].isin([2012, 2013, 2014, 2015])
].copy()

validation_data = model_data[
    model_data["SEASON"] == 2016
].copy()

test_data = model_data[
    model_data["SEASON"] == 2017
].copy()

In [67]:
print("Training:", train_data.shape)
print("Validation:", validation_data.shape)
print("Test:", test_data.shape)

Training: (5681, 41)
Validation: (1405, 41)
Test: (1382, 41)


In [68]:
print("\nTraining seasons:")
print(train_data["SEASON"].value_counts().sort_index())

print("\nValidation season:")
print(validation_data["SEASON"].value_counts().sort_index())

print("\nTest season:")
print(test_data["SEASON"].value_counts().sort_index())


Training seasons:
SEASON
2012    1420
2013    1427
2014    1418
2015    1416
Name: count, dtype: int64

Validation season:
SEASON
2016    1405
Name: count, dtype: int64

Test season:
SEASON
2017    1382
Name: count, dtype: int64


In [69]:
feature_columns = [
    col for col in model_data.columns
    if col.startswith("HOME_") or col.startswith("AWAY_")
]

print("Number of features:", len(feature_columns))
print(feature_columns)

Number of features: 21
['HOME_TEAM_ID', 'HOME_TEAM_WINS', 'HOME_LAST3_POINTS_SCORED', 'HOME_LAST3_POINTS_ALLOWED', 'HOME_LAST3_FG_PCT', 'HOME_LAST3_FT_PCT', 'HOME_LAST3_FG3_PCT', 'HOME_LAST3_AST', 'HOME_LAST3_REB', 'HOME_LAST3_POINT_DIFFERENTIAL', 'HOME_DAYS_SINCE_LAST_GAME', 'AWAY_TEAM_ID', 'AWAY_LAST3_POINTS_SCORED', 'AWAY_LAST3_POINTS_ALLOWED', 'AWAY_LAST3_FG_PCT', 'AWAY_LAST3_FT_PCT', 'AWAY_LAST3_FG3_PCT', 'AWAY_LAST3_AST', 'AWAY_LAST3_REB', 'AWAY_LAST3_POINT_DIFFERENTIAL', 'AWAY_DAYS_SINCE_LAST_GAME']


In [70]:
X_train = train_data[feature_columns]
y_train = train_data["TOTAL_POINTS"]

X_validation = validation_data[feature_columns]
y_validation = validation_data["TOTAL_POINTS"]

X_test = test_data[feature_columns]
y_test = test_data["TOTAL_POINTS"]

In [71]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (5681, 21)
y_train: (5681,)
X_validation: (1405, 21)
y_validation: (1405,)
X_test: (1382, 21)
y_test: (1382,)


In [72]:
print("Number of features:", len(feature_columns))
print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)

Number of features: 21
X_train: (5681, 21)
X_validation: (1405, 21)
X_test: (1382, 21)


In [73]:
print(feature_columns)

['HOME_TEAM_ID', 'HOME_TEAM_WINS', 'HOME_LAST3_POINTS_SCORED', 'HOME_LAST3_POINTS_ALLOWED', 'HOME_LAST3_FG_PCT', 'HOME_LAST3_FT_PCT', 'HOME_LAST3_FG3_PCT', 'HOME_LAST3_AST', 'HOME_LAST3_REB', 'HOME_LAST3_POINT_DIFFERENTIAL', 'HOME_DAYS_SINCE_LAST_GAME', 'AWAY_TEAM_ID', 'AWAY_LAST3_POINTS_SCORED', 'AWAY_LAST3_POINTS_ALLOWED', 'AWAY_LAST3_FG_PCT', 'AWAY_LAST3_FT_PCT', 'AWAY_LAST3_FG3_PCT', 'AWAY_LAST3_AST', 'AWAY_LAST3_REB', 'AWAY_LAST3_POINT_DIFFERENTIAL', 'AWAY_DAYS_SINCE_LAST_GAME']


In [74]:
feature_columns = [
    "HOME_LAST3_POINTS_SCORED",
    "HOME_LAST3_POINTS_ALLOWED",
    "HOME_LAST3_FG_PCT",
    "HOME_LAST3_FT_PCT",
    "HOME_LAST3_FG3_PCT",
    "HOME_LAST3_AST",
    "HOME_LAST3_REB",
    "HOME_LAST3_POINT_DIFFERENTIAL",
    "HOME_DAYS_SINCE_LAST_GAME",

    "AWAY_LAST3_POINTS_SCORED",
    "AWAY_LAST3_POINTS_ALLOWED",
    "AWAY_LAST3_FG_PCT",
    "AWAY_LAST3_FT_PCT",
    "AWAY_LAST3_FG3_PCT",
    "AWAY_LAST3_AST",
    "AWAY_LAST3_REB",
    "AWAY_LAST3_POINT_DIFFERENTIAL",
    "AWAY_DAYS_SINCE_LAST_GAME"
]

print("Number of features:", len(feature_columns))
print(feature_columns)

Number of features: 18
['HOME_LAST3_POINTS_SCORED', 'HOME_LAST3_POINTS_ALLOWED', 'HOME_LAST3_FG_PCT', 'HOME_LAST3_FT_PCT', 'HOME_LAST3_FG3_PCT', 'HOME_LAST3_AST', 'HOME_LAST3_REB', 'HOME_LAST3_POINT_DIFFERENTIAL', 'HOME_DAYS_SINCE_LAST_GAME', 'AWAY_LAST3_POINTS_SCORED', 'AWAY_LAST3_POINTS_ALLOWED', 'AWAY_LAST3_FG_PCT', 'AWAY_LAST3_FT_PCT', 'AWAY_LAST3_FG3_PCT', 'AWAY_LAST3_AST', 'AWAY_LAST3_REB', 'AWAY_LAST3_POINT_DIFFERENTIAL', 'AWAY_DAYS_SINCE_LAST_GAME']


In [75]:
X_train = train_data[feature_columns]
y_train = train_data["TOTAL_POINTS"]

X_validation = validation_data[feature_columns]
y_validation = validation_data["TOTAL_POINTS"]

X_test = test_data[feature_columns]
y_test = test_data["TOTAL_POINTS"]

In [76]:
print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)

X_train: (5681, 18)
X_validation: (1405, 18)
X_test: (1382, 18)
